# Week 2 RAG Prototype: Reranking & Context Selection

**Student task:** implement and evaluate the reranking and context-selection stages of a Retrieval-Augmented Generation (RAG) quiz-generation pipeline.

## What this notebook does
1. Accepts candidate chunks from an existing hybrid retriever (or a `chunks.json` file).
2. Reranks candidates for a query using a cross-encoder.
3. Selects a compact set of relevant, non-redundant passages under a configurable context budget.
4. Preserves document, page, and chunk references.
5. Exports `reranked_candidates.json`, `selected_context.json`, and a short run summary.

**Important:** the notebook does not invent content from the four PDFs. For a real run, upload your actual `chunks.json` and, if available, your hybrid retrieval output. The built-in sample below is only a clearly labelled smoke test, not evidence from your PDFs.


## 1. Install dependencies

Run this cell in Google Colab. The cross-encoder model is downloaded the first time it is used. If the model cannot load, the notebook falls back to a lexical relevance scorer so the rest of the workflow can still be tested.


In [ ]:
!pip -q install sentence-transformers scikit-learn pandas

## 2. Configure the run

Edit `QUERY` to a question answerable from your corpus. The query “What is Retrieval-Augmented Generation?” is a good first test for the RAG paper. You can also test a Transformer/self-attention or hashing question if those topics are in your chunks.


In [ ]:
from pathlib import Path
import json, re, os, math, textwrap
import pandas as pd

QUERY = "What is Retrieval-Augmented Generation?"
TOP_N_CANDIDATES = 10       # candidates sent to reranking
FINAL_CONTEXT_CHUNKS = 4    # target number of selected chunks
MAX_CONTEXT_WORDS = 900     # simple, transparent context budget
MIN_RELEVANCE_SCORE = None  # e.g. 0.15, or None to disable threshold
USE_DEMO_DATA = True        # set False after uploading your real files

OUTPUT_DIR = Path("/content/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Query:", QUERY)
print("Output directory:", OUTPUT_DIR)


## 3. Load the candidate chunks

Upload `chunks.json` (from your existing chunking step) and optionally a JSON file containing hybrid retrieval results. The loader accepts common JSON layouts: a list of chunks, `{"chunks": [...]}`, `{"results": [...]}`, or `{"candidates": [...]}`. It normalizes metadata fields but keeps the original fields too.

If a hybrid-results file is supplied, it is preferred because reranking should operate on retrieved candidates rather than the whole corpus. If not, the notebook can score chunks from `chunks.json` as a practical fallback.


In [ ]:
from google.colab import files

uploaded = files.upload()  # choose chunks.json and optionally hybrid_results.json
uploaded_names = list(uploaded.keys())
print("Uploaded:", uploaded_names)

def read_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def unwrap_records(obj):
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ("candidates", "results", "chunks", "documents", "data"):
            if isinstance(obj.get(key), list):
                return obj[key]
    raise ValueError("Could not find a list of records in this JSON file.")

def normalize_record(rec, idx):
    if not isinstance(rec, dict):
        rec = {"text": str(rec)}
    text = (
        rec.get("text") or rec.get("chunk_text") or rec.get("content")
        or rec.get("page_text") or rec.get("passage") or rec.get("document")
        or ""
    )
    meta = rec.get("metadata") if isinstance(rec.get("metadata"), dict) else {}
    doc_id = rec.get("doc_id", rec.get("document_id", meta.get("doc_id", meta.get("document_id", "unknown"))))
    page = rec.get("page", rec.get("page_number", meta.get("page", meta.get("page_number", None))))
    chunk_id = rec.get("chunk_id", meta.get("chunk_id", f"chunk_{idx}"))
    heading = rec.get("heading", meta.get("heading", ""))
    rank = rec.get("rank", rec.get("hybrid_rank", idx + 1))
    score = rec.get("score", rec.get("hybrid_score", rec.get("rrf_score", None)))
    normalized = dict(rec)
    normalized.update({
        "text": str(text).strip(),
        "doc_id": str(doc_id),
        "page": page,
        "chunk_id": str(chunk_id),
        "heading": str(heading or ""),
        "input_rank": rank,
        "input_score": score,
    })
    return normalized

json_names = [n for n in uploaded_names if n.lower().endswith(".json")]
hybrid_candidates = None
chunk_records = None

for name in json_names:
    raw = read_json_file(name)
    records = [normalize_record(r, i) for i, r in enumerate(unwrap_records(raw))]
    lower = name.lower()
    if any(k in lower for k in ("hybrid", "retrieval", "candidate", "result")):
        hybrid_candidates = records
    elif "chunk" in lower:
        chunk_records = records

if hybrid_candidates:
    source_records = hybrid_candidates
    SOURCE_USED = "uploaded hybrid retrieval candidates"
elif chunk_records:
    source_records = chunk_records
    SOURCE_USED = "uploaded chunks.json (fallback: all chunks are candidates)"
else:
    source_records = None
    SOURCE_USED = None

print("Candidate source:", SOURCE_USED or "demo sample (if enabled)")
print("Records loaded:", len(source_records) if source_records else 0)


## 4. Demo data fallback (smoke test only)

If you have not uploaded compatible JSON yet and `USE_DEMO_DATA=True`, the notebook uses generic illustrative snippets to check that the code runs. These are **not quotations or verified passages from your PDFs**. For your submitted results, rerun with your actual corpus files.


In [ ]:
if not source_records and USE_DEMO_DATA:
    source_records = [
        normalize_record({
            "doc_id": "DEMO_ONLY_RAG", "page": 1, "chunk_id": "demo_rag_1",
            "heading": "RAG overview",
            "text": "Retrieval-augmented generation combines a language model with an external retrieval step. Relevant passages are retrieved and supplied as context to help ground the generated response."
        }, 0),
        normalize_record({
            "doc_id": "DEMO_ONLY_TRANSFORMER", "page": 2, "chunk_id": "demo_transformer_1",
            "heading": "Self-attention",
            "text": "Self-attention lets each token compute a representation using information from other tokens in the same sequence."
        }, 1),
        normalize_record({
            "doc_id": "DEMO_ONLY_HASHING", "page": 3, "chunk_id": "demo_hash_1",
            "heading": "Hashing",
            "text": "A hash function maps keys to positions in a table. A collision occurs when distinct keys map to the same position."
        }, 2),
        normalize_record({
            "doc_id": "DEMO_ONLY_RAG", "page": 2, "chunk_id": "demo_rag_2",
            "heading": "Why retrieve context",
            "text": "A retrieval system can return passages relevant to a user query. The generator can use the selected evidence to produce an answer and cite the source passage."
        }, 3),
        normalize_record({
            "doc_id": "DEMO_ONLY_MISC", "page": 4, "chunk_id": "demo_misc_1",
            "heading": "Unrelated example",
            "text": "A chemical reaction network can represent species and reactions, with edges or transitions connecting the elements of the system."
        }, 4),
    ]
    SOURCE_USED = "demo sample — NOT corpus evidence"

if not source_records:
    raise ValueError("No chunks loaded. Upload chunks.json or enable USE_DEMO_DATA.")

source_records = [r for r in source_records if r.get("text")]
print(f"Ready: {len(source_records)} non-empty records from {SOURCE_USED}")
pd.DataFrame([{
    "doc_id": r["doc_id"], "page": r["page"], "chunk_id": r["chunk_id"],
    "heading": r["heading"], "text_preview": r["text"][:180]
} for r in source_records]).head(10)


## 5. Retrieve a candidate shortlist (only if hybrid results were not supplied)

If hybrid retrieval candidates were uploaded, those are used directly. Otherwise, this lightweight lexical pre-filter selects a shortlist from all chunks using TF-IDF cosine similarity. This is only a fallback; for the assignment comparison, use the actual hybrid retrieval output when available.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def lexical_prefilter(query, records, top_n=10):
    texts = [r["text"] for r in records]
    if len(texts) == 1:
        return records[:top_n]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
    try:
        matrix = vectorizer.fit_transform(texts + [query])
        sims = cosine_similarity(matrix[-1], matrix[:-1]).ravel()
    except ValueError:
        sims = [0.0] * len(records)
    order = sorted(range(len(records)), key=lambda i: float(sims[i]), reverse=True)[:top_n]
    output = []
    for rank, i in enumerate(order, start=1):
        item = dict(records[i])
        item["prefilter_score"] = float(sims[i])
        item["prefilter_rank"] = rank
        output.append(item)
    return output

if SOURCE_USED and SOURCE_USED.startswith("uploaded hybrid"):
    candidates = source_records[:TOP_N_CANDIDATES]
    prefilter_method = "uploaded hybrid candidates"
else:
    candidates = lexical_prefilter(QUERY, source_records, TOP_N_CANDIDATES)
    prefilter_method = "TF-IDF lexical fallback"

print("Candidate shortlist method:", prefilter_method)
print("Shortlisted candidates:", len(candidates))


## 6. Rerank candidates with a cross-encoder

A cross-encoder reads the query and each candidate passage together and assigns a relevance score. Unlike bi-encoder retrieval, it can jointly attend to query and passage tokens, making it useful for ordering a relatively small shortlist.

Model used: `cross-encoder/ms-marco-MiniLM-L-6-v2`. If loading/inference fails, the code falls back to TF-IDF scores and records which method was used.


In [ ]:
RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
rerank_method = None

try:
    from sentence_transformers import CrossEncoder
    cross_encoder = CrossEncoder(RERANKER_NAME)
    pairs = [(QUERY, rec["text"]) for rec in candidates]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    rerank_method = f"CrossEncoder: {RERANKER_NAME}"
    for rec, score in zip(candidates, scores):
        rec["rerank_score"] = float(score)
except Exception as exc:
    print("Cross-encoder unavailable; using TF-IDF fallback. Reason:", repr(exc)[:300])
    fallback = lexical_prefilter(QUERY, candidates, len(candidates))
    fallback_scores = {r["chunk_id"]: r.get("prefilter_score", 0.0) for r in fallback}
    for rec in candidates:
        rec["rerank_score"] = float(fallback_scores.get(rec["chunk_id"], 0.0))
    rerank_method = "TF-IDF fallback (not a cross-encoder)"

reranked = sorted(candidates, key=lambda r: r.get("rerank_score", float("-inf")), reverse=True)
for rank, rec in enumerate(reranked, start=1):
    rec["rerank_rank"] = rank

print("Reranking method:", rerank_method)
pd.DataFrame([{
    "rerank_rank": r["rerank_rank"],
    "rerank_score": round(r["rerank_score"], 4),
    "doc_id": r["doc_id"], "page": r["page"], "chunk_id": r["chunk_id"],
    "heading": r["heading"], "text_preview": r["text"][:160]
} for r in reranked])


## 7. Select context: relevance + redundancy + budget

Selection uses a simple, explainable greedy strategy:
- start from the highest reranked score;
- reject candidates below `MIN_RELEVANCE_SCORE` when a threshold is configured;
- avoid near-duplicate passages using TF-IDF cosine similarity;
- stop at `FINAL_CONTEXT_CHUNKS` or `MAX_CONTEXT_WORDS`.

The output preserves `doc_id`, `page`, `chunk_id`, `heading`, the text, and the reranking score. Word count is used as a transparent approximation to a token budget; it is not an exact tokenizer-based token count.


In [ ]:
REDUNDANCY_SIMILARITY_THRESHOLD = 0.82

def word_count(text):
    return len(re.findall(r"\b\w+\b", text or ""))

def select_context(query, ranked_records, max_chunks=4, max_words=900,
                   min_score=None, redundancy_threshold=0.82):
    selected = []
    selected_texts = []
    used_words = 0
    all_texts = [r["text"] for r in ranked_records]
    try:
        vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
        all_matrix = vectorizer.fit_transform(all_texts)
    except Exception:
        vectorizer, all_matrix = None, None

    for idx, rec in enumerate(ranked_records):
        score = float(rec.get("rerank_score", 0.0))
        if min_score is not None and score < min_score:
            continue
        words = word_count(rec["text"])
        if words == 0 or used_words + words > max_words:
            continue

        is_redundant = False
        if selected and vectorizer is not None and all_matrix is not None:
            candidate_vec = all_matrix[idx]
            for chosen in selected:
                chosen_idx = next((j for j, x in enumerate(ranked_records)
                                   if x.get("chunk_id") == chosen.get("chunk_id")), None)
                if chosen_idx is not None:
                    sim = float(cosine_similarity(candidate_vec, all_matrix[chosen_idx])[0, 0])
                    if sim >= redundancy_threshold:
                        is_redundant = True
                        break
        if is_redundant:
            continue

        selected.append(rec)
        selected_texts.append(rec["text"])
        used_words += words
        if len(selected) >= max_chunks:
            break

    result = []
    for pos, rec in enumerate(selected, start=1):
        result.append({
            "selection_rank": pos,
            "doc_id": rec.get("doc_id", "unknown"),
            "page": rec.get("page"),
            "chunk_id": rec.get("chunk_id"),
            "heading": rec.get("heading", ""),
            "text": rec["text"],
            "rerank_score": rec.get("rerank_score"),
            "rerank_rank": rec.get("rerank_rank"),
            "word_count": word_count(rec["text"]),
        })
    return result

selected_context = select_context(
    QUERY, reranked,
    max_chunks=FINAL_CONTEXT_CHUNKS,
    max_words=MAX_CONTEXT_WORDS,
    min_score=MIN_RELEVANCE_SCORE,
    redundancy_threshold=REDUNDANCY_SIMILARITY_THRESHOLD
)

print(f"Selected {len(selected_context)} chunks; total words = {sum(x['word_count'] for x in selected_context)}")
pd.DataFrame(selected_context)


## 8. Save outputs with source traceability

These JSON files can be consumed by the next stage (quiz generation). The selected context file includes the original query and the methods/configuration used.


In [ ]:
def dump_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

reranked_path = OUTPUT_DIR / "reranked_candidates.json"
selected_path = OUTPUT_DIR / "selected_context.json"
summary_path = OUTPUT_DIR / "run_summary.json"

dump_json(reranked_path, {
    "query": QUERY,
    "source_used": SOURCE_USED,
    "candidate_method": prefilter_method,
    "reranking_method": rerank_method,
    "candidates": [
        {
            **r,
            "text": r.get("text", "")
        } for r in reranked
    ]
})

dump_json(selected_path, {
    "query": QUERY,
    "selection_method": "greedy relevance-ranked selection with TF-IDF redundancy filtering and word budget",
    "reranking_method": rerank_method,
    "context_budget_words": MAX_CONTEXT_WORDS,
    "selected_chunk_count": len(selected_context),
    "selected_context": selected_context
})

summary = {
    "query": QUERY,
    "source_used": SOURCE_USED,
    "candidate_shortlist_method": prefilter_method,
    "reranking_method": rerank_method,
    "candidates_reranked": len(reranked),
    "chunks_selected": len(selected_context),
    "selected_word_count": sum(x["word_count"] for x in selected_context),
    "max_context_words": MAX_CONTEXT_WORDS,
    "redundancy_similarity_threshold": REDUNDANCY_SIMILARITY_THRESHOLD,
    "selected_chunk_ids": [x["chunk_id"] for x in selected_context],
}
dump_json(summary_path, summary)
print(json.dumps(summary, indent=2))
print("Saved:", reranked_path, selected_path, summary_path)


## 9. Inspect the final context

Review this output before passing it to quiz generation. If the selected evidence does not answer the query, do not generate a question from it; record that evidence is insufficient and try a better query or inspect retrieval.


In [ ]:
for item in selected_context:
    print("=" * 88)
    print(f"Selection rank: {item['selection_rank']} | score: {item['rerank_score']}")
    print(f"Source: {item['doc_id']} | page: {item['page']} | chunk: {item['chunk_id']}")
    if item["heading"]:
        print("Heading:", item["heading"])
    print(textwrap.fill(item["text"], width=100))

if not selected_context:
    print("No context selected. Check the query, score threshold, and context budget.")


## 10. Download the outputs

The three files below are written in Colab's `/content/results/` folder. Run this cell to download them to your computer.


In [ ]:
from google.colab import files

for path in [reranked_path, selected_path, summary_path]:
    files.download(str(path))


## 11. Short analysis for the assignment report

Use the actual results from your corpus, not the demo sample, to complete this table. Reranking and context selection should be evaluated on whether the final passages support the query and reduce irrelevant/redundant context.

| Check | What to report |
|---|---|
| Query | The exact query tested |
| Input candidates | Number and source (hybrid output preferred) |
| Reranker | Cross-encoder model or fallback method |
| Selected context | Number of chunks and word budget used |
| Traceability | Whether every selected chunk has a valid document/page/chunk reference |
| Relevance | Do the selected passages answer the query? Give a brief evidence-based explanation |
| Redundancy | Were near-duplicate passages removed? |
| Limitation | Cross-encoder model download/runtime, word count as approximate token budget, and dependency on upstream retrieval quality |

### Design rationale
- **Reranking** improves the order of a small candidate set by scoring query–passage pairs together.
- **Context selection** limits the generator input to a few high-ranked, non-redundant passages.
- **Source metadata** is retained so quiz answers can be traced to the original document, page, and chunk.
- **Failure handling:** if the evidence is weak or absent, the downstream generator should abstain instead of inventing an answer.

### Before submission
- [ ] Run the notebook with your actual `chunks.json`.
- [ ] Prefer the actual hybrid retrieval output over the TF-IDF fallback.
- [ ] Use a query answerable from your current PDFs.
- [ ] Check that selected references match the source files.
- [ ] Include the generated `selected_context.json` and `run_summary.json` with your submission.
- [ ] State clearly if the fallback scorer was used; do not claim a cross-encoder result unless it actually ran.
